In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------------
# 0) LOAD DATA
# --------------------------------------------------------
df = pd.read_csv(r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_input_files\RX1_small.csv")
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df = df.sort_values("Date").set_index("Date")

price = df["PX_CLOSE_1D"].astype(float)
stdev_price = df["st_dev"].astype(float)

# --------------------------------------------------------
# 1) PARAMETERS
# --------------------------------------------------------
START_NAV_USD = 10_000_000
ANNUAL_VOL = 0.20
FX = 0.6485
tickValue = 10
tickSize = 0.01
MULT = tickValue / tickSize     # 1000
LOOKBACK = 36
DECAY = 2 / (LOOKBACK + 1)      # 2 / 37

# --------------------------------------------------------
# 2) RETURNS (two kinds)
# --------------------------------------------------------
# for variance (your sheet)
df["ret_net_for_var"] = price.diff()

# for forecast PnL check
df["ret_pct_for_sig"] = price.pct_change()

# --------------------------------------------------------
# 3) EWMA VARIANCE on NET RETURNS (robust start)
# --------------------------------------------------------
variance = [np.nan] * len(df)
ret_net = df["ret_net_for_var"].values
idxs = list(range(len(df)))

# find first valid net return (non-NaN, non-zero)
start_idx = None
for i in idxs:
    rn = ret_net[i]
    if rn is not None and not np.isnan(rn) and rn != 0:
        start_idx = i
        break

if start_idx is not None:
    first_sq = ret_net[start_idx] ** 2
    variance[start_idx] = first_sq
    prev_var = first_sq
    for i in range(start_idx + 1, len(df)):
        rn = ret_net[i]
        if rn is None or np.isnan(rn):
            rn2 = 0.0
        else:
            rn2 = rn * rn
        v = DECAY * rn2 + (1 - DECAY) * prev_var
        variance[i] = v
        prev_var = v

df["variance"] = variance
df["stdev_variance"] = np.sqrt(df["variance"])

# --------------------------------------------------------
# 4) 4/16 EWMAs on PRICE
# --------------------------------------------------------
df["ewma_fast"] = price.ewm(span=4, adjust=False).mean()
df["ewma_slow"] = price.ewm(span=16, adjust=False).mean()
df["raw_crossover"] = df["ewma_fast"] - df["ewma_slow"]

# --------------------------------------------------------
# 5) VOL-ADJ CROSSOVER + FORECAST
# --------------------------------------------------------
df["vol_adj_crossover"] = df["raw_crossover"] / df["stdev_variance"]
df["scaled_forecast"] = df["vol_adj_crossover"] * 7.5
df["capped_forecast"] = df["scaled_forecast"].clip(-20, 20)
df["forecast_x_return"] = df["capped_forecast"].shift(1) * df["ret_pct_for_sig"]

# --------------------------------------------------------
# 6) PRECOMPUTE PIECES FOR SIZING (these don’t depend on NAV)
# --------------------------------------------------------
df["price_volatility"] = stdev_price / price * 100
df["one_pct_move"] = price * 0.01
df["block_value_eur"] = df["one_pct_move"] * MULT
df["icv_eur"] = df["price_volatility"] * df["block_value_eur"]
df["ivv"] = df["icv_eur"] / FX   # EUR -> USD

# --------------------------------------------------------
# 7) ITERATIVE LOOP FOR:
#    - dynamic NAV
#    - dynamic daily cash vol target
#    - positions, trades
#    - carry / trade PnL
# --------------------------------------------------------
nav_list = []
daily_cash_vol_list = []
target_contracts_list = []
trades_list = []
carry_pnl_eur_list = []
trade_pnl_eur_list = []
carry_pnl_usd_list = []
trade_pnl_usd_list = []
pnl_usd_list = []
strategy_ret_list = []

# we also need yesterday's position to compute carry PnL
prev_target_contracts = 0
prev_nav = START_NAV_USD

prices = price.values
ivv_vals = df["ivv"].values
capped = df["capped_forecast"].values

for i in range(len(df)):
    # 1) today's dynamic daily vol target (based on *yesterday* NAV)
    daily_cash_vol_target = prev_nav * ANNUAL_VOL / np.sqrt(256)

    # 2) today's volatility scaler
    ivv_today = ivv_vals[i]
    if np.isnan(ivv_today) or ivv_today == 0:
        vol_scaler_today = np.nan
    else:
        vol_scaler_today = daily_cash_vol_target / ivv_today

    # 3) today's subsystem position (before rounding & lag)
    cap_f = capped[i]
    if np.isnan(cap_f) or np.isnan(vol_scaler_today):
        subsystem_pos = np.nan
    else:
        subsystem_pos = (cap_f * vol_scaler_today) / 10.0

    # 4) we trade the LAGGED rounded position:
    #    so today's target is yesterday's subsystem_pos rounded
    if i == 0:
        target_contracts = np.nan  # first day no trade
    else:
        # round yesterday's subsystem_pos
        y_sub = (capped[i-1] * (daily_cash_vol_list[-1] / ivv_vals[i-1]) / 10.0
                 if not np.isnan(ivv_vals[i-1]) else np.nan)
        # but we actually stored yesterday's vol scaler?
        # instead of recomputing, better: store yesterday target in list
        # since we already have a running prev_target_contracts, we can use it
        target_contracts = round(prev_target_contracts)

    # but we want to update prev_target_contracts at the *end* of loop
    # to make carry PnL easy, let's define today's traded position like this:
    if i == 0:
        trades = 0
        target_contracts_eff = 0
    else:
        # we decide that today's effective target is the rounded subsystem_pos from TODAY
        # (simpler and clearer)
        target_contracts_eff = 0 if np.isnan(subsystem_pos) else int(round(subsystem_pos))
        trades = target_contracts_eff - prev_target_contracts

    # 5) price info
    px_today = prices[i]
    if i == 0:
        px_yday = np.nan
    else:
        px_yday = prices[i - 1]
    delta_price = np.nan if i == 0 else (px_today - px_yday)

    # 6) carry PnL (on yesterday's position)
    if i == 0 or np.isnan(delta_price):
        carry_pnl_eur = 0.0
    else:
        carry_pnl_eur = prev_target_contracts * delta_price * MULT

    # 7) trade PnL (on today's trades, using exec = yesterday's close)
    if i == 0 or np.isnan(px_yday):
        trade_pnl_eur = 0.0
    else:
        trade_pnl_eur = trades * (px_yday - px_today) * MULT

    # 8) convert to USD (consistent with ivv = icv/FX)
    carry_pnl_usd = carry_pnl_eur / FX
    trade_pnl_usd = trade_pnl_eur / FX
    pnl_usd = carry_pnl_usd + trade_pnl_usd

    # 9) update NAV
    nav_today = prev_nav + pnl_usd

    # 10) daily return
    strat_ret = pnl_usd / prev_nav if prev_nav != 0 else np.nan

    # 11) store
    nav_list.append(nav_today)
    daily_cash_vol_list.append(daily_cash_vol_target)
    target_contracts_list.append(target_contracts_eff)
    trades_list.append(trades)
    carry_pnl_eur_list.append(carry_pnl_eur)
    trade_pnl_eur_list.append(trade_pnl_eur)
    carry_pnl_usd_list.append(carry_pnl_usd)
    trade_pnl_usd_list.append(trade_pnl_usd)
    pnl_usd_list.append(pnl_usd)
    strategy_ret_list.append(strat_ret)

    # 12) update prev for next loop
    prev_nav = nav_today
    prev_target_contracts = target_contracts_eff

# --------------------------------------------------------
# 8) ASSIGN BACK TO DF
# --------------------------------------------------------
df["nav_usd"] = nav_list
df["daily_cash_vol_target"] = daily_cash_vol_list
df["target_contracts"] = target_contracts_list
df["trades"] = trades_list
df["carry_pnl_eur"] = carry_pnl_eur_list
df["trade_pnl_eur"] = trade_pnl_eur_list
df["carry_pnl_usd"] = carry_pnl_usd_list
df["trade_pnl_usd"] = trade_pnl_usd_list
df["pnl_usd"] = pnl_usd_list
df["strategy_ret"] = strategy_ret_list
df["tickValue"] = tickValue
df["tickSize"] = tickSize

# --------------------------------------------------------
# 9) SHARPE (on realized, dynamic-NAV returns)
# --------------------------------------------------------
valid = df["strategy_ret"].replace([np.inf, -np.inf], np.nan).dropna()
if not valid.empty and valid.std() != 0:
    sharpe = valid.mean() / valid.std() * np.sqrt(252)
else:
    sharpe = np.nan
df["sharpe_usd"] = sharpe

# --------------------------------------------------------
# 10) SAVE
# --------------------------------------------------------
out_path = r"C:\Users\loci_\Desktop\trading_webapp\DATA\gpt_output_rx1_full_chain_with_dynNAV.csv"
df.to_csv(out_path, index=True)
print("✅ saved to:", out_path)


✅ saved to: C:\Users\loci_\Desktop\trading_webapp\DATA\gpt_output_rx1_full_chain_with_dynNAV.csv
